# Paired synthetic BioASQ generator-model comparison

This notebook compares two configurations on the **same 20 training-only PubMed sources**:

- **Mini → Mini:** `gpt-4.1-mini-2025-04-14` generates and verifies.
- **GPT-4.1 → Mini:** `gpt-4.1-2025-04-14` generates; `gpt-4.1-mini-2025-04-14` verifies.

The first 20 deterministic manifest entries contain 10 clean and 10 distractor sources. Each generator proposes two questions per source, giving 40 paired question slots per arm. Prompts, source IDs, evidence-arm assignment, seed, verifier, and final filters are fixed; only the generator changes.

Run each section in order. API responses are cached independently for each arm.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
PYTHON = Path('/home/dinh-tuan/miniconda3/envs/bioasq/bin/python')
RUNNER = PROJECT_ROOT / 'cse_dpo/run_synthetic_factoid_qa_model_comparison.py'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/synthetic_factoid_qa/gpt41mini_vs_gpt41_generation_20sources_v2'

def run_phase(phase, *, dry_run=False):
    cmd = [
        str(PYTHON), str(RUNNER),
        '--phase', phase,
        '--output-root', str(OUTPUT_ROOT),
        '--python', str(PYTHON),
    ]
    if dry_run:
        cmd.append('--dry-run')
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

print('Output root:', OUTPUT_ROOT)


## 1. Prepare identical inputs

This is local and makes **no API calls**. The runner prepares both arms independently and aborts if their manifests, eligible source pools, source order, or 10/10 evidence-arm balance differ.


In [ ]:
run_phase('prepare')


In [ ]:
config = json.loads((OUTPUT_ROOT / 'comparison_config.json').read_text())
config


In [ ]:
manifest = pd.read_json(OUTPUT_ROOT / 'mini_mini/source_manifest.jsonl', lines=True).head(20)
print(manifest.groupby(['split', 'evidence_arm']).size())
manifest[['synthetic_source_id', 'split', 'evidence_arm']].head()


## 2. Preview prompts

This prints the first generation prompt for each arm and makes **no API calls**.


In [ ]:
run_phase('generate', dry_run=True)


## 3. Generate questions

This makes up to 20 calls per arm. Each call asks for two questions, so each arm targets 40 candidate questions. Successful results are cached.


In [ ]:
run_phase('generate')


In [ ]:
for arm in ('mini_mini', 'gpt41_mini'):
    path = OUTPUT_ROOT / arm / 'generation_summary.json'
    print(arm, json.loads(path.read_text()))


## 4. Blinded verification

Both arms use exactly the same GPT-4.1 Mini verifier. It receives the generated questions and final evidence packs without seeing the intended answers.


In [ ]:
run_phase('verify')


In [ ]:
for arm in ('mini_mini', 'gpt41_mini'):
    path = OUTPUT_ROOT / arm / 'verification_summary.json'
    print(arm, json.loads(path.read_text()))


## 5. Finalize and compare

Finalization and comparison are local. Generation must pass the v2 list, metadata, leakage, answer-type, and relation-diversity checks. Verification must cite the target resource and confirm a unique explicit relation. The intended answer may exactly match the verifier output or be wholly contained in a longer verifier clause; reverse containment is rejected.


In [ ]:
run_phase('finalize')
run_phase('compare')


In [ ]:
summary = json.loads((OUTPUT_ROOT / 'comparison_summary.json').read_text())
pd.DataFrame({
    arm: {
        'generator': values['generator_model'],
        'verifier': values['verifier_model'],
        'expected_slots': values['expected_question_count'],
        'generated': values['generated_and_finalized_question_count'],
        'generation_failed': values['generation_failed_question_count'],
        'source_ineligible': values.get('source_ineligible_question_count', 0),
        'verification_failed': values['verification_failed_question_count'],
        'accepted': values['accepted_count'],
        'acceptance_rate': values['acceptance_rate_over_expected_slots'],
    }
    for arm, values in summary['arms'].items()
}).T


In [ ]:
pd.Series(summary['paired_acceptance_outcomes'], name='question_slots')


## 6. Blind manual audit

Acceptance rate alone cannot establish biomedical quality. Review the paired rows without assuming GPT-4.1 is better. Fill `manual_preference` with `mini`, `gpt41`, `tie`, or `neither`, and use `manual_notes` for factual, ambiguity, boundary, or distractor errors.


In [ ]:
review_path = OUTPUT_ROOT / 'paired_manual_review.csv'
review = pd.read_csv(review_path)
review[[
    'synthetic_question_id', 'evidence_arm', 'paired_outcome',
    'mini_question', 'mini_answer', 'mini_accepted',
    'gpt41_question', 'gpt41_answer', 'gpt41_accepted',
    'manual_preference', 'manual_notes',
]]


Selection rule: choose the stronger generator only if the blinded manual audit shows fewer unsupported, ambiguous, answer-leaking, incorrectly scoped, or distractor-sensitive questions. A higher automatic acceptance rate by itself is insufficient.
